# Convert raw data to 'strict' json

In [5]:
import json
import gzip
import os
dataset_name = "Beauty"
os.makedirs(dataset_name, exist_ok=True)

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.dumps(eval(l))

# Beauty dataset
f = open(f"./{dataset_name}/{dataset_name}.json", 'w')
for l in parse(f"reviews_{dataset_name}_5.json.gz"):
  f.write(l + '\n')

In [6]:
# print the number of lines in the file and the first line
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')
print("Number of lines:", sum(1 for _ in data))
data.seek(0)  # Reset file pointer to the beginning
print("First line:", data.readline().strip())
data.close()

Number of lines: 198502
First line: {"reviewerID": "A1YJEY40YUW4SE", "asin": "7806397051", "reviewerName": "Andrea", "helpful": [3, 4], "reviewText": "Very oily and creamy. Not at all what I expected... ordered this to try to highlight and contour and it just looked awful!!! Plus, took FOREVER to arrive.", "overall": 1.0, "summary": "Don't waste your money", "unixReviewTime": 1391040000, "reviewTime": "01 30, 2014"}


In [1]:
%cd <your path>\TIGER-main\data

In [2]:
# 👈👈👈
dataset_name = "Beauty"

import numpy as np
import pandas as pd
import json

# Initialize mapping dictionaries
userID_mapping = {}
itemID_mapping = {}

# Open the JSON file for reading
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')

# Initialize lists to store userID, itemID, and timestamp
userIDs = []
itemIDs = []
timestamps = []

# Process each line in the JSON file
for line in data:
    review = json.loads(line.strip())
    userID = review['reviewerID']
    itemID = review['asin']
    timestamp = review['unixReviewTime']
    
    # Map userID to an integer starting from 1
    if userID not in userID_mapping:
        userID_mapping[userID] = len(userID_mapping) + 1
    
    # Map itemID to an integer starting from 1
    if itemID not in itemID_mapping:
        itemID_mapping[itemID] = len(itemID_mapping) + 1
    
    # Append mapped values and timestamp to lists
    userIDs.append(userID_mapping[userID])
    itemIDs.append(itemID_mapping[itemID])
    timestamps.append(timestamp)

# Save mapping dictionaries as .npy files
np.save(f'./{dataset_name}/user_mapping.npy', userID_mapping)
print("user_num:", len(userID_mapping))
print("the first five userID mapping:", list(userID_mapping.items())[:5])
np.save(f'./{dataset_name}/item_mapping.npy', itemID_mapping)
print("item_num:", len(itemID_mapping))
print("the first five itemID mapping:", list(itemID_mapping.items())[:5])

# Group itemIDs by userID and sort by timestamp
user_item_mapping = {}
for userID, itemID, timestamp in zip(userIDs, itemIDs, timestamps):
    if userID not in user_item_mapping:
        user_item_mapping[userID] = []
    user_item_mapping[userID].append((itemID, timestamp))

# Sort itemIDs for each user by timestamp
for userID in user_item_mapping:
    user_item_mapping[userID].sort(key=lambda x: x[1])
    user_item_mapping[userID] = [item[0] for item in user_item_mapping[userID]]

# Print a sample of the results
print("user-item mapping:", list(user_item_mapping.items())[:5])

# Split data into training, validation, and testing sets using leave-one-out strategy
train_data = {}
val_data = {}
test_data = {}

for userID, item_sequence in user_item_mapping.items():
    # Assign the last item for testing, the second-to-last for validation, and the rest for training
    train_data[userID] = item_sequence[:-2]
    val_data[userID] = item_sequence[:-1]
    test_data[userID] = item_sequence

# Print a sample of the split data
# print("training data:", list(train_data.items())[:5])
# print("validation data:", list(val_data.items())[:5])
# print("testing data:", list(test_data.items())[:5])

# Prepare data for train, validation, and test sets
def prepare_data(data_dict):
    rows = []
    for userID, item_sequence in data_dict.items():
        history = item_sequence[:-1]
        target = item_sequence[-1]
        rows.append({'user': userID, 'history': history, 'target': target})
    return pd.DataFrame(rows)

# Create dataframes for train, validation, and test sets
train_df = prepare_data(train_data)
print("\nTraining data shape:", train_df.shape)
print("the first 3 rows of training data:\n", train_df.head(3))
val_df = prepare_data(val_data)
print("\nValidation data shape:", val_df.shape)
print("the first 3 rows of validation data:\n", val_df.head(3))
test_df = prepare_data(test_data)
print("\nTesting data shape:", test_df.shape)
print("the first 3 rows of testing data:\n", test_df.head(3))

# Save dataframes to parquet files
train_df.to_parquet(f'./{dataset_name}/train.parquet', index=False)
val_df.to_parquet(f'./{dataset_name}/valid.parquet', index=False)
test_df.to_parquet(f'./{dataset_name}/test.parquet', index=False)

print("Data saved to parquet files.")

data.close()


user_num: 22363
the first five userID mapping: [('A1YJEY40YUW4SE', 1), ('A60XNB876KYML', 2), ('A3G6XNM240RMWA', 3), ('A1PQFP6SAJ6D80', 4), ('A38FVHZTNQ271F', 5)]
item_num: 12101
the first five itemID mapping: [('7806397051', 1), ('9759091062', 2), ('9788072216', 3), ('9790790961', 4), ('9790794231', 5)]
user-item mapping: [(1, [6846, 7873, 4585, 1, 5406]), (2, [816, 10406, 11194, 11651, 9716, 1, 233]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863, 6609]), (4, [5522, 439, 5161, 11140, 1, 7849]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390])]

Training data shape: (22363, 3)
the first 3 rows of training data:
    user                           history  target
0     1                      [6846, 7873]    4585
1     2        [816, 10406, 11194, 11651]    9716
2     3  [1, 6050, 7977, 5252, 4211, 243]   11204

Validation data shape: (22363, 3)
the first 3 rows of validation data:
    user                                  history  target
0     1                       [684

# Generate Item Semantic Embeddings

In [3]:
# 👈👈👈
# Open the metadata file for reading

with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    # Initialize a dictionary to store the extracted information
    item_info = {}

    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')

        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                # 1. text
                # 'xxxxxxx'
                'title': metadata.get('title') if metadata.get('title') else None,
                # 190
                'price': metadata.get('price') if metadata.get('price') else None,
                # {'Beauty': 10486, "xxx": xxx}
                'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
                # 'COKA'
                'brand': metadata.get('brand') if metadata.get('brand') else None,
                # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
                'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
                # 'xxxxxxxxxxxxxxxx'
                'description': metadata.get('description') if metadata.get('description') else None,

                # 2. image
                # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
                'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
            }
        # asin = metadata.get('asin')
        #
        # # Check if the asin exists in the reverse mapping
        # if asin in reverse_itemID_mapping.values():
        #     # for k in metadata.keys():
        #     #     print(k, ": ", metadata[k])
        #     # break

# Print the information for the first 5 items
for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'price': 5.04, 'salesRank': {'Beauty': 10486}, 'brand': 'COKA', 'categories': ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers'], 'description': 'An extensive range of 15 multiple vibrant long wear concealer colour with different skin tones to create more than 10,000 amazing looks. Using the most commonly applied shades, ensures the best skin colour match and guarantees a traceless and natural finish. Enabling layering and mixing, provides total camouflage for almost any skin problem including blemishes, scars, birthmarks and black circles. It is also suitable to use as bronzer. The light colour is suitable for redness, acne and so on. The medium colour is perfect for dark shadows in the under-eye area. The dark colour provides exceptional camouflage and adheres well to the skin. Silky glossy colour and high quality ingredients together to care skin around and can

In [5]:
# Prepare data for embedding
item_embeddings = []
for itemID, info in item_info.items():
    # Combine relevant fields into a single text for embedding

    item_embeddings.append({
        'ItemID': itemID,
        'title': info.get('title', ''),
        'price': info.get('price', ''),
        'salesRank': info.get('salesRank', ''),
        'brand': info.get('brand', ''),
        'categories': info.get('categories', ''),
        'image': info.get('imUrl', ''),
        'description': info.get('description', ''),
    })

# Convert to DataFrame
item_emb_df = pd.DataFrame(item_embeddings)

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))


Item embeddings DataFrame shape: (12100, 8)
The first 3 rows of item embeddings DataFrame:
    ItemID                                              title  price  \
0       1  WAWO 15 Color Professionl Makeup Eyeshadow Cam...   5.04   
1       2                  Xtreme Brite Brightening Gel 1oz.  19.99   
2       3  Prada Candy By Prada Eau De Parfum Spray 1.7 O...  65.86   

           salesRank         brand  \
0  {'Beauty': 10486}          COKA   
1  {'Beauty': 52254}  Xtreme Brite   
2  {'Beauty': 78916}         Prada   

                                          categories  \
0  [Beauty, Makeup, Face, Concealers & Neutralizers]   
1  [Beauty, Hair Care, Styling Products, Creams, ...   
2        [Beauty, Fragrance, Women's, Eau de Parfum]   

                                               image  \
0  http://ecx.images-amazon.com/images/I/41Rn18Oe...   
1  http://ecx.images-amazon.com/images/I/41QWW9v1...   
2  http://ecx.images-amazon.com/images/I/51iT2k6L...   

                   

In [1]:
import transformers
print(transformers.__version__)

C:\Users\sakaixue\AppData\Local\JetBrains\PyCharm2025.1\demo\PyCharmLearningProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.51.3


In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    'iic/gme-Qwen2-VL-2B-Instruct',
    cache_dir='<your_path>/TIGER-main/models/gme_qwen2vl'   # 👈 自定义目录
)
print("模型下载目录:", model_dir)

In [6]:
# =====================================
# 🧩 Part 1: Download Images for GME-Qwen2-VL
# =====================================

import os
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

# ======== 图片缓存目录 ========
IMG_DIR = "./cached_images"
os.makedirs(IMG_DIR, exist_ok=True)

def download_image(url, item_id):
    """下载图片并缓存"""
    if not url or not isinstance(url, str) or not url.startswith("http"):
        return None
    local_path = os.path.join(IMG_DIR, f"{item_id}.jpg")
    if os.path.exists(local_path):
        return local_path
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img.save(local_path)
        return local_path
    except Exception as e:
        print(f"[WARN] Failed to download {url}: {e}")
        return None

# ======== 1️⃣ 下载阶段 ========
print("Downloading images...")

for idx, row in tqdm(item_emb_df.iterrows(), total=len(item_emb_df), desc="Image download"):
    info = row["Info"] if "Info" in row else row
    url = info.get("imUrl", row.get("image", None))
    download_image(url, row["ItemID"])

print(f"✅ All available images cached in: {IMG_DIR}")

Image download: 100%|██████████| 12100/12100 [09:48<00:00, 20.55it/s]

✅ All available images cached in: ./cached_images


In [11]:
# =====================================
# 🧩 Resize existing images to 128x128
# =====================================

import os
from tqdm import tqdm
from PIL import Image

# 输入输出目录
SRC_DIR = "./cached_images"        # 你已经爬好的图片文件夹
DST_DIR = "./cached_images_128"    # 统一 128x128 后保存到这里
os.makedirs(DST_DIR, exist_ok=True)

def resize_and_save(src_path, dst_path, size=(128, 128)):
    try:
        img = Image.open(src_path).convert("RGB")
        img = img.resize(size, Image.Resampling.LANCZOS)
        img.save(dst_path, format="JPEG", quality=90)
        return True
    except Exception as e:
        print(f"[WARN] Failed to resize {src_path}: {e}")
        return False

# 遍历所有图片文件
img_files = [f for f in os.listdir(SRC_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"Found {len(img_files)} images to resize.")

for fname in tqdm(img_files, desc="Resizing"):
    src_path = os.path.join(SRC_DIR, fname)
    dst_path = os.path.join(DST_DIR, fname)
    if not os.path.exists(dst_path):  # 避免重复
        resize_and_save(src_path, dst_path)

print(f"✅ All images resized to 128x128 and saved in: {DST_DIR}")

Found 12093 images to resize.


Resizing: 100%|██████████| 12093/12093 [00:23<00:00, 514.04it/s]

✅ All images resized to 128x128 and saved in: ./cached_images_128


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from modelscope import AutoModel
from transformers.utils.versions import require_version
import torch
from PIL import Image
import os
import warnings
from io import BytesIO

import gc  # ✅ 新增导入

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # ✅ 禁止 tokenizer 并行警告

# ========== 0️⃣ 版本提示 ==========
require_version(
    "transformers<4.52.0",
    "The remote code has some issues with transformers>=4.52.0, please downgrade: pip install transformers==4.51.3"
)

# ========== 1️⃣ 基础设置 ==========
dataset_name = "Beauty"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

print("Loading Alibaba-NLP/gme-Qwen2-VL-2B-Instruct ...")
gme = AutoModel.from_pretrained(
    "/root/autodl-tmp/TIGER/model/gme_qwen2vl/iic/gme-Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map=device,
    trust_remote_code=True
)

# ========== 自动根据显存调整 batch size ==========
# if torch.cuda.is_available():
#     mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
#     if mem_gb < 8:
#         BATCH_SIZE = 4
#     elif mem_gb < 12:
#         BATCH_SIZE = 8
#     elif mem_gb < 24:
#         BATCH_SIZE = 16
#     else:
#         BATCH_SIZE = 32
# else:
#     BATCH_SIZE = 2
BATCH_SIZE = 16
print(f"Auto batch size: {BATCH_SIZE}")

# ========== 2️⃣ 图片缓存目录 ==========
IMG_DIR = "./cached_images_128/cached_images_128"
os.makedirs(IMG_DIR, exist_ok=True)

def load_cached_image(item_id, url=None):
    """优先从缓存加载，否则跳过"""
    local_path = os.path.join(IMG_DIR, f"{item_id}.jpg")
    if os.path.exists(local_path):
        try:
            return Image.open(local_path).convert("RGB").resize((128, 128))
        except:
            return None
    elif url and url.startswith("http"):
        try:
            response = requests.get(url, timeout=5)
            img = Image.open(BytesIO(response.content)).convert("RGB").resize((128, 128))
            img.save(local_path)
            return img
        except:
            return None
    return None

def get_text(info):
    """组装文本字段"""
    raw_title = str(info.get("title", "")).strip()
    text = f"title: {raw_title}"
    fields = [
        ("categories", info.get("categories", "")),
        ("price", info.get("price", "")),
        ("salesRank", info.get("salesRank", "")),
        ("brand", info.get("brand", "")),
        ("description", info.get("description", "")),
    ]
    for k, v in fields:
        text += f"\n{k}: {v}"
    return text.strip() if text.strip() else None

# ========== 3️⃣ 批量生成 embedding ==========
embs = []
n_items = len(item_emb_df)
print(f"Encoding {n_items} items in batches of {BATCH_SIZE} with GME-Qwen2-VL-2B-Instruct...")

for i in tqdm(range(0, n_items, BATCH_SIZE), desc="GME batch encoding"):
    batch = item_emb_df.iloc[i:i+BATCH_SIZE]

    texts, images = [], []
    for _, row in batch.iterrows():
        info = row["Info"] if "Info" in row else row
        text = get_text(info)
        img = load_cached_image(row["ItemID"], info.get("imUrl", None) or row.get("image", None))
        texts.append(text)
        images.append(img)

    # 构造输入
    with torch.no_grad():
        try:
            # 如果 batch 中所有样本都有图 + 文 → 直接 fused
            if all(texts) and all(images):
                emb = gme.get_fused_embeddings(texts=texts, images=images)
            # 只有文本
            elif all(texts) and not any(images):
                emb = gme.get_text_embeddings(texts=texts)
            # 只有图片
            elif all(images) and not any(texts):
                emb = gme.get_image_embeddings(images=images)
            else:
                # 混合情况：逐个处理
                sub_embs = []
                for t, img in zip(texts, images):
                    if t and img:
                        e = gme.get_fused_embeddings(texts=[t], images=[img])
                    elif t:
                        e = gme.get_text_embeddings(texts=[t])
                    elif img:
                        e = gme.get_image_embeddings(images=[img])
                    else:
                        e = torch.zeros((1, 1024), device=device)
                    sub_embs.append(e)
                emb = torch.cat(sub_embs, dim=0)

            # 模型内部已归一化，不需再手动除 norm
            for e in emb:
                embs.append(e.cpu().numpy().tolist())

        except Exception as e:
            print(f"[WARN] Batch {i//BATCH_SIZE} failed: {e}")
            for _ in range(len(batch)):
                embs.append(np.zeros(1024).tolist())

    # ✅ 每个 batch 后清理 GPU 显存与 Python 对象
    # del emb, texts, images
    gc.collect()
    torch.cuda.empty_cache()

print("✅ Embedding generation completed.")

# ========== 4️⃣ 保存 ==========
item_emb_df_ = item_emb_df.copy()
item_emb_df_["embedding"] = embs

In [9]:
SAVE_PATH = f"./{dataset_name}/item_gmeqwen2vl_emb.parquet"
item_emb_df_.to_parquet(SAVE_PATH, index=False)
print(f"💾 Embedding saved to {SAVE_PATH}")

In [ ]:
item_emb_df_.head(5)